# Module 7B Applied Lab — SI Demo
## Extractive QA Pipeline: TriviaQA `rc.nocontext` Evaluation

**Goal:** Build an end-to-end QA evaluation pipeline, measure performance with Exact Match and Token-F1, and diagnose three canonical failure modes.

**Dataset:** TriviaQA `rc.nocontext` validation split, first 150 examples  
**Model:** `deepset/roberta-base-squad2` (RoBERTa fine-tuned on SQuAD 2.0)  
**Estimated time:** ~90 minutes

| Section | Topic | Time |
|---|---|---|
| 1 | Setup & Imports | 5 min |
| 2 | Pipeline construction + one-passage inference | 10 min |
| 3 | Evaluation harness (normalization, EM, token-F1) | 15 min |
| 4 | Run pipeline + harness over dataset | 10 min |
| 5 | Inspect aggregate metrics | 10 min |
| 6 | Three failure modes | 15 min |
| 7 | Pivot to learner assignment | 15 min |

---
## Section 1 — Setup & Imports

Install the four required packages and import every module the lab uses.  
Run this cell once; the `%pip install` is idempotent (safe to re-run).

In [1]:
# ── install once if running in a fresh environment ──────────────────────────
%pip install -q datasets transformers torch pandas             # install core libs

# ── standard library ────────────────────────────────────────────────────────
import string                                                  # punctuation chars
import re                                                      # regex engine
from collections import Counter                                # token frequency

# ── third-party ─────────────────────────────────────────────────────────────
import torch                                                   # tensor ops
import pandas as pd                                            # tabular data ops
from datasets import load_dataset                              # HuggingFace datasets
from transformers import (                                     # HF model components
    AutoModelForQuestionAnswering,
    AutoTokenizer,
)

print("All imports successful.")                               # confirm environment

Note: you may need to restart the kernel to use updated packages.


/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)
/home/brandon/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


---
## Section 2 — QA Pipeline Construction + One-Passage Inference

We instantiate a `pipeline("question-answering")` backed by a RoBERTa model fine-tuned on SQuAD 2.0, then run one example end-to-end so we can inspect the full result dictionary.

> **Design note — why pass the question as context?**  
> `rc.nocontext` has no passage. The `pipeline()` API requires a `context` string.  
> By passing the question itself, we observe what the model extracts when  
> the answer is *not present in the text* — a pure **knowledge-gap** scenario.  
> This is the central failure mode of no-context TriviaQA.

In [2]:
# ── 2a. Build the pipeline ───────────────────────────────────────────────────
_MODEL_NAME = "deepset/roberta-base-squad2"                    # SQuAD2-tuned RoBERTa
_tokenizer  = AutoTokenizer.from_pretrained(_MODEL_NAME)       # load tokenizer
_model      = AutoModelForQuestionAnswering.from_pretrained(   # load model
    _MODEL_NAME
)
_model.eval()                                                  # inference mode

def qa_pipeline(question, context, **kwargs):                  # drop-in pipeline shim
    inputs = _tokenizer(                                       # tokenize pair
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    with torch.no_grad():                                      # no gradient needed
        outputs = _model(**inputs)
    start_logits = outputs.start_logits[0]                     # shape: (seq_len,)
    end_logits   = outputs.end_logits[0]                       # shape: (seq_len,)
    start_idx = torch.argmax(start_logits).item()              # best start position
    # end must be >= start; slice then offset
    end_idx = (
        torch.argmax(end_logits[start_idx:]).item() + start_idx
    )
    answer_ids = inputs.input_ids[0][start_idx : end_idx + 1]  # token ids of span
    answer = _tokenizer.decode(answer_ids, skip_special_tokens=True)  # to string
    # score: joint probability of the chosen span boundaries
    score = (
        torch.softmax(start_logits, dim=0)[start_idx].item()
        * torch.softmax(end_logits, dim=0)[end_idx].item()
    )
    return {"answer": answer, "score": score,                  # match pipeline output
            "start": start_idx, "end": end_idx}

# ── 2b. Load the TriviaQA subset ────────────────────────────────────────────
dataset = load_dataset(                                        # fetch from HF Hub
    "trivia_qa",                                               # dataset identifier
    "rc.nocontext",                                            # no-passage config
    split="validation[:150]",                                  # first 150 examples
    trust_remote_code=True,                                    # allow dataset script
)

print(f"Loaded {len(dataset)} examples.")                      # confirm dataset size
print(f"Column names: {dataset.column_names}")                 # show available fields

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 162.47it/s]
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'trivia_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Generating test split: 100%|██████████| 17210/17210 [00:00<00:00, 308420.52 examples/s]

Loaded 150 examples.
Column names: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer']


In [3]:
# ── 2c. Inspect one example ──────────────────────────────────────────────────
example = dataset[0]                                           # grab the first record
print("Question  :", example["question"])                      # show question text
print("Gold value:", example["answer"]["value"])               # canonical answer
print("Aliases   :", example["answer"]["aliases"][:5], "...")  # first 5 valid aliases

Question  : Who was the man behind The Chipmunks?
Gold value: David Seville
Aliases   : ['David Seville'] ...


In [4]:
# ── 2d. Run one-passage inference ────────────────────────────────────────────
result = qa_pipeline(                                          # run the pipeline
    question=example["question"],                              # the trivia question
    context=example["question"],                               # question = context
)

print("Result dictionary:")                                    # label the output
for key, val in result.items():                                # unpack each field
    print(f"  {key:8s}: {val}")                               # key: value aligned

print()
print("Gold answer :", example["answer"]["value"])             # compare gold
print("Predicted   :", result["answer"])                       # compare prediction
print("Confidence  :", round(result["score"], 4))              # model certainty

Result dictionary:
  answer  : 
  score   : 0.7941683290140666
  start   : 0
  end     : 0

Gold answer : David Seville
Predicted   : 
Confidence  : 0.7942


---
## Section 3 — Evaluation Harness: Normalization, EM, and Token-F1

Every QA leaderboard since SQuAD 2016 uses this same three-step harness:

| Function | What it does |
|---|---|
| `normalize_answer` | Lowercase → strip punctuation → strip articles → collapse whitespace |
| `exact_match` | Returns 1 if the normalized prediction equals any normalized gold alias |
| `token_f1` | Precision/recall/F1 over shared token counts; best score across all aliases |

Each normalization step is demonstrated with a **worked example** at the end of the cell.

In [5]:
# ── 3a. Normalization ────────────────────────────────────────────────────────
def normalize_answer(text):                                    # SQuAD-style normalizer
    text = text.lower()                                        # step 1: lowercase
    text = text.translate(                                     # step 2: strip punctuation
        str.maketrans("", "", string.punctuation)              #   build delete table
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)               # step 3: drop articles
    text = " ".join(text.split())                              # step 4: collapse whitespace
    return text                                                # return cleaned string


# ── 3b. Exact Match ──────────────────────────────────────────────────────────
def exact_match(prediction, gold_answers):                     # binary correctness metric
    pred_norm = normalize_answer(prediction)                   # normalize prediction
    for gold in gold_answers:                                  # try every valid alias
        if pred_norm == normalize_answer(gold):                # compare normalized forms
            return 1                                           # match found → EM = 1
    return 0                                                   # no match → EM = 0


# ── 3c. Token-F1 ─────────────────────────────────────────────────────────────
def token_f1(prediction, gold_answers):                        # token-overlap F1 metric
    pred_tokens = normalize_answer(prediction).split()         # tokenize prediction
    if not pred_tokens:                                        # guard: empty prediction
        return 0.0                                             # return 0 immediately
    best_f1 = 0.0                                              # accumulate best F1
    for gold in gold_answers:                                  # try every valid alias
        gold_tokens = normalize_answer(gold).split()           # tokenize gold answer
        if not gold_tokens:                                    # guard: empty gold alias
            continue                                           # skip this alias
        common  = Counter(pred_tokens) & Counter(gold_tokens)  # shared token counts
        n_common = sum(common.values())                        # total shared tokens
        if n_common == 0:                                      # no overlap → skip
            continue
        precision = n_common / len(pred_tokens)                # correct / predicted
        recall    = n_common / len(gold_tokens)                # correct / gold
        f1 = 2 * precision * recall / (precision + recall)     # harmonic mean
        best_f1 = max(best_f1, f1)                             # keep best alias F1
    return best_f1                                             # return best F1


print("Functions defined: normalize_answer, exact_match, token_f1")  # confirm

Functions defined: normalize_answer, exact_match, token_f1


In [6]:
# ── 3d. Worked normalization example ────────────────────────────────────────
# Why does normalization matter? Walk through each step.

raw_pred = "The United States of America"                     # raw prediction string
raw_gold = "USA"                                              # canonical gold answer

print("── Raw strings ──")
print(f"  Prediction : '{raw_pred}'")                         # before normalization
print(f"  Gold       : '{raw_gold}'")                         # before normalization

print("── After lowercase ──")
print(f"  Prediction : '{raw_pred.lower()}'")                 # lowercase only
print(f"  Gold       : '{raw_gold.lower()}'")                 # lowercase only

print("── After full normalization ──")
print(f"  Prediction : '{normalize_answer(raw_pred)}'")       # all four steps
print(f"  Gold       : '{normalize_answer(raw_gold)}'")       # all four steps

print("── Metrics ──")
print(f"  EM   (no alias) : {exact_match(raw_pred, [raw_gold])}")         # EM = 0
print(f"  F1   (no alias) : {token_f1(raw_pred, [raw_gold]):.3f}")        # F1 = 0
print(f"  F1  (with alias 'United States') : "
      f"{token_f1(raw_pred, [raw_gold, 'United States']):.3f}")  # F1 > 0 with alias

── Raw strings ──
  Prediction : 'The United States of America'
  Gold       : 'USA'
── After lowercase ──
  Prediction : 'the united states of america'
  Gold       : 'usa'
── After full normalization ──
  Prediction : 'united states of america'
  Gold       : 'usa'
── Metrics ──
  EM   (no alias) : 0
  F1   (no alias) : 0.000
  F1  (with alias 'United States') : 0.667


---
## Section 4 — Run Pipeline + Harness over the Full Subset

Loop over all 150 examples, score each prediction with EM and token-F1,  
and write the results to `triviaqa_results.csv` for later inspection.

> **Note:** This cell takes ~2–5 minutes on CPU. The pipeline processes one example at a time.

In [7]:
records = []                                                   # list of result dicts

for i, ex in enumerate(dataset):                               # iterate all examples
    question     = ex["question"]                              # question text
    gold_value   = ex["answer"]["value"]                       # canonical answer
    gold_aliases = list(ex["answer"]["aliases"])               # all valid aliases
    if gold_value not in gold_aliases:                         # avoid duplicates
        gold_aliases.append(gold_value)                        # include canonical form

    try:
        out   = qa_pipeline(question=question, context=question)  # run inference
        pred  = out["answer"]                                  # predicted span text
        score = out["score"]                                   # model confidence
    except Exception:                                          # guard pipeline errors
        pred  = ""                                             # fallback: empty string
        score = 0.0                                            # fallback: zero score

    em = exact_match(pred, gold_aliases)                       # compute EM (0 or 1)
    f1 = token_f1(pred, gold_aliases)                          # compute token-F1

    records.append({                                           # accumulate results
        "question":   question,                                # question text
        "gold":       gold_value,                              # canonical gold answer
        "prediction": pred,                                    # model output span
        "score":      round(score, 4),                         # confidence (4 dp)
        "em":         em,                                      # exact match (0 or 1)
        "f1":         round(f1, 4),                            # token-F1 (0.0–1.0)
    })

    if (i + 1) % 25 == 0:                                     # progress every 25 rows
        print(f"  Processed {i + 1} / {len(dataset)} examples...")  # show progress

df = pd.DataFrame(records)                                     # convert list to DataFrame
df.to_csv("triviaqa_results.csv", index=False)                 # persist to disk
print(f"\nDone — {len(df)} rows saved to triviaqa_results.csv")  # confirm save
df.head(3)                                                     # preview first 3 rows

  Processed 25 / 150 examples...
  Processed 50 / 150 examples...
  Processed 75 / 150 examples...
  Processed 100 / 150 examples...
  Processed 125 / 150 examples...
  Processed 150 / 150 examples...

Done — 150 rows saved to triviaqa_results.csv


,question,gold,prediction,score,em,f1
0,Who was the man behind The Chipmunks?,David Seville,,0.7942,0,0.0
1,Which Lloyd Webber musical premiered in the US...,Sunset Boulevard,,0.8595,0,0.0
2,Who was the next British Prime Minister after ...,Campbell-Bannerman,,0.7366,0,0.0


---
## Section 5 — Inspect Aggregate Metrics

Compute headline EM and F1, interpret the gap between them, and sort by F1 ascending to surface the worst predictions.

**Key concept — the EM–F1 gap:**

| Gap size | What it means |
|---|---|
| Large (> 0.05) | Many near-misses: surface-form / normalization failures dominant |
| Small (≤ 0.05) | Mostly binary right/wrong: knowledge-gap or distractor failures dominant |

In a no-context TriviaQA run the gap is typically **small** — predictions are fragments of the question, not near-correct answer strings.

In [8]:
mean_em = df["em"].mean()                                      # aggregate EM rate
mean_f1 = df["f1"].mean()                                      # aggregate F1 rate
gap     = mean_f1 - mean_em                                    # F1 minus EM gap

print(f"Mean EM   : {mean_em:.3f}")                            # print EM percentage
print(f"Mean F1   : {mean_f1:.3f}")                            # print F1 percentage
print(f"F1–EM gap : {gap:.3f}")                                # print gap
print()

if gap > 0.05:                                                 # threshold: 5 pp
    print("Interpretation: large gap → surface-form / normalization failures present")  # analysis
else:
    print("Interpretation: small gap → knowledge-gap failures dominate")  # analysis

Mean EM   : 0.000
Mean F1   : 0.000
F1–EM gap : 0.000

Interpretation: small gap → knowledge-gap failures dominate


In [9]:
# ── Sort by F1 ascending to find worst predictions ───────────────────────────
worst10 = df.sort_values("f1").head(10)                        # 10 lowest-F1 examples

print("── Worst 10 predictions (sorted by F1 ascending) ──")
display(worst10[[                                               # subset relevant columns
    "question", "gold", "prediction", "score", "em", "f1"       # cols for failure analysis
]])

── Worst 10 predictions (sorted by F1 ascending) ──


,question,gold,prediction,score,em,f1
0,Who was the man behind The Chipmunks?,David Seville,,0.7942,0,0.0
1,Which Lloyd Webber musical premiered in the US...,Sunset Boulevard,,0.8595,0,0.0
2,Who was the next British Prime Minister after ...,Campbell-Bannerman,,0.7366,0,0.0
3,Who had a 70s No 1 hit with Kiss You All Over?,Exile,,0.9621,0,0.0
4,What claimed the life of singer Kathleen Ferrier?,Cancer,,0.9043,0,0.0
5,Rita Coolidge sang the title song for which Bo...,Octopussy,,0.8474,0,0.0
6,What was the last US state to reintroduce alco...,Utah,,0.7130,0,0.0
7,Which actress was voted Miss Greenwich Village...,Lauren Bacall,,0.7702,0,0.0
8,What is the Japanese share index called?,Nikkei,,0.8386,0,0.0
9,What was the name of Michael Jackson's autobio...,Moonwalk,,0.6463,0,0.0


In [10]:
# ── Score distribution ────────────────────────────────────────────────────────
print("── F1 distribution ──")
print(df["f1"].describe().round(3))                            # count, mean, std, quartiles
print()
print("── EM breakdown ──")
print(df["em"].value_counts().rename({0: "Wrong (EM=0)", 1: "Correct (EM=1)"}))  # counts

── F1 distribution ──
count    150.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: f1, dtype: float64

── EM breakdown ──
em
Wrong (EM=0)    150
Name: count, dtype: int64


---
## Section 6 — Three TriviaQA Failure Modes

For each failure mode we follow the same three-step structure:
1. **Name the pattern** — what is happening mechanically?
2. **Show concrete examples** — real rows from the dataset.
3. **State the production implication** — what would a team actually do?

These three modes serve as a **template** for the learner assignment —  
not to be copied, but to illustrate the depth of analysis expected.

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# FAILURE MODE 1 — Knowledge-Gap Failures
# ════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("FAILURE MODE 1: Knowledge-Gap Failures")
print("=" * 65)
print("""
Pattern : The question requires world knowledge the model does not
          have. With no passage, the model extracts a span from the
          question text itself — which almost never contains the
          answer. Confidence (score) can be moderate; the answer is
          wrong regardless.

Why it happens : rc.nocontext exposes the model's parametric recall
          directly. Post-cutoff facts, niche trivia, and low-frequency
          entities are all absent from the model's weights.
""")

# Filter: confident-but-wrong, near-zero token overlap
kg_mask = (                                                    # define filter
    (df["em"] == 0) &                                          # wrong answer
    (df["f1"] < 0.3) &                                         # near-zero overlap
    (df["score"] > 0.3)                                        # model is confident
)
kg_examples = df[kg_mask].head(3)                              # show top 3 examples

print(f"Found {kg_mask.sum()} knowledge-gap candidates. Showing top 3:\n")  # count

for idx, row in kg_examples.iterrows():                        # iterate selected rows
    print(f"Q    : {row['question']}")                         # question text
    print(f"Gold : {row['gold']}")                             # correct answer
    print(f"Pred : '{row['prediction']}'  (score={row['score']})")  # model output
    print(f"       EM={row['em']}  F1={row['f1']}")            # both near zero
    print("-" * 65)

print("""
Production implication:
  → Add retrieval (RAG): fetch a Wikipedia passage before running
    the extractive reader. Without context, recall is bounded by
    the model's parametric knowledge at its training cutoff.
  → Monitor score distribution: knowledge-gap failures tend to
    cluster at lower confidence, but not always.
""")

FAILURE MODE 1: Knowledge-Gap Failures

Pattern : The question requires world knowledge the model does not
          have. With no passage, the model extracts a span from the
          question text itself — which almost never contains the
          answer. Confidence (score) can be moderate; the answer is
          wrong regardless.

Why it happens : rc.nocontext exposes the model's parametric recall
          directly. Post-cutoff facts, niche trivia, and low-frequency
          entities are all absent from the model's weights.

Found 146 knowledge-gap candidates. Showing top 3:

Q    : Who was the man behind The Chipmunks?
Gold : David Seville
Pred : ''  (score=0.7942)
       EM=0  F1=0.0
-----------------------------------------------------------------
Q    : Which Lloyd Webber musical premiered in the US on 10th December 1993?
Gold : Sunset Boulevard
Pred : ''  (score=0.8595)
       EM=0  F1=0.0
-----------------------------------------------------------------
Q    : Who was the n

In [12]:
# ════════════════════════════════════════════════════════════════════════════
# FAILURE MODE 2 — Surface-Form / Normalization Failures
# ════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("FAILURE MODE 2: Surface-Form / Normalization Failures")
print("=" * 65)
print("""
Pattern : The prediction is semantically correct but uses a different
          surface form from any gold alias. After normalization,
          EM = 0 but F1 > 0 (partial token overlap).
          Classic examples: 'USA' vs 'United States',
          '1985' vs 'nineteen eighty-five'.

Signal  : A large EM–F1 gap in the aggregate metrics indicates this
          failure mode is active.
""")

# Filter: wrong by EM but partial token overlap
sf_mask = (df["em"] == 0) & (df["f1"] > 0.4)                  # surface-form filter
sf_examples = df[sf_mask].head(3)                              # top 3 examples

print(f"Found {sf_mask.sum()} surface-form candidates. Showing top 3:\n")  # count

for _, row in sf_examples.iterrows():                          # iterate selected rows
    print(f"Q    : {row['question']}")                         # question text
    print(f"Gold : {row['gold']}")                             # canonical gold
    print(f"Pred : '{row['prediction']}'")                     # model output
    print(f"       EM={row['em']}  F1={row['f1']:.3f}")        # F1 > 0, EM = 0
    print("-" * 65)

# ── Worked normalization example ─────────────────────────────────────────────
print("\n── Worked normalization example ──")
ex_pred = "United States of America"                           # example prediction
ex_gold = "USA"                                                # example gold
print(f"Prediction : '{ex_pred}'")                             # raw prediction
print(f"Gold       : '{ex_gold}'")                             # raw gold
print(f"norm(pred) : '{normalize_answer(ex_pred)}'")           # after normalization
print(f"norm(gold) : '{normalize_answer(ex_gold)}'")           # after normalization
print(f"EM : {exact_match(ex_pred, [ex_gold])}")               # still 0
print(f"F1 with alias 'United States' : "
      f"{token_f1(ex_pred, [ex_gold, 'United States']):.3f}")  # F1 > 0 with alias

print("""
Production implication:
  → Expand alias lists using Wikidata Q-item labels (they carry
    many valid surface forms for the same entity).
  → Replace string-EM with an NLI-based semantic equivalence
    model for more robust evaluation.
""")

FAILURE MODE 2: Surface-Form / Normalization Failures

Pattern : The prediction is semantically correct but uses a different
          surface form from any gold alias. After normalization,
          EM = 0 but F1 > 0 (partial token overlap).
          Classic examples: 'USA' vs 'United States',
          '1985' vs 'nineteen eighty-five'.

Signal  : A large EM–F1 gap in the aggregate metrics indicates this
          failure mode is active.

Found 0 surface-form candidates. Showing top 3:


── Worked normalization example ──
Prediction : 'United States of America'
Gold       : 'USA'
norm(pred) : 'united states of america'
norm(gold) : 'usa'
EM : 0
F1 with alias 'United States' : 0.667

Production implication:
  → Expand alias lists using Wikidata Q-item labels (they carry
    many valid surface forms for the same entity).
  → Replace string-EM with an NLI-based semantic equivalence
    model for more robust evaluation.



In [13]:
# ════════════════════════════════════════════════════════════════════════════
# FAILURE MODE 3 — Distractor-Pull Failures
# ════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("FAILURE MODE 3: Distractor-Pull Failures")
print("=" * 65)
print("""
Pattern : The question contains a salient near-answer — a famous
          name, year, or place — and the model latches onto it
          instead of the correct (often rarer) answer.
          The model's confidence (score) can be misleadingly high.

Key lesson: score alone is NOT a reliable acceptance signal.
          A high-confidence wrong answer is more dangerous than a
          low-confidence wrong answer in a production system.
""")

# Heuristic: short predictions (likely a grabbed entity), confident, wrong
dp_mask = (                                                    # define filter
    (df["em"] == 0) &                                          # wrong answer
    (df["prediction"].str.split().str.len() <= 2) &            # short (1–2 tokens)
    (df["score"] > 0.5)                                        # high confidence
)
dp_examples = df[dp_mask].head(3)                              # top 3 examples

print(f"Found {dp_mask.sum()} distractor-pull candidates. Showing top 3:\n")  # count

for _, row in dp_examples.iterrows():                          # iterate selected rows
    print(f"Q    : {row['question']}")                         # question text
    print(f"Gold : {row['gold']}")                             # correct answer
    print(f"Pred : '{row['prediction']}'  (score={row['score']:.3f})")  # confident + wrong
    print(f"       EM={row['em']}  F1={row['f1']:.3f}")        # both near zero
    print("-" * 65)

print("""
Production implication:
  → Calibrate confidence: compute score–EM calibration plots
    across confidence buckets. If score > 0.8 rows have EM < 0.5,
    the model is overconfident and score cannot be used as a
    threshold for automated acceptance in production.
  → Add a confidence-calibration layer (temperature scaling or
    Platt scaling) before exposing score to downstream systems.
""")

FAILURE MODE 3: Distractor-Pull Failures

Pattern : The question contains a salient near-answer — a famous
          name, year, or place — and the model latches onto it
          instead of the correct (often rarer) answer.
          The model's confidence (score) can be misleadingly high.

Key lesson: score alone is NOT a reliable acceptance signal.
          A high-confidence wrong answer is more dangerous than a
          low-confidence wrong answer in a production system.

Found 141 distractor-pull candidates. Showing top 3:

Q    : Who was the man behind The Chipmunks?
Gold : David Seville
Pred : ''  (score=0.794)
       EM=0  F1=0.000
-----------------------------------------------------------------
Q    : Which Lloyd Webber musical premiered in the US on 10th December 1993?
Gold : Sunset Boulevard
Pred : ''  (score=0.860)
       EM=0  F1=0.000
-----------------------------------------------------------------
Q    : Who was the next British Prime Minister after Arthur Balfour?
G

In [14]:
# ── Failure mode summary table ────────────────────────────────────────────────
summary = pd.DataFrame({                                       # build summary DataFrame
    "Failure Mode": [                                          # mode names
        "Knowledge-Gap",
        "Surface-Form",
        "Distractor-Pull",
    ],
    "Filter": [                                                # filter description
        "EM=0, F1<0.3, score>0.3",
        "EM=0, F1>0.4",
        "EM=0, pred len≤2, score>0.5",
    ],
    "Count": [                                                 # example counts
        kg_mask.sum(),                                         # knowledge-gap count
        sf_mask.sum(),                                         # surface-form count
        dp_mask.sum(),                                         # distractor-pull count
    ],
    "Signal": [                                                # what to look for
        "Confident wrong prediction, no token overlap",
        "Large EM–F1 gap in aggregate metrics",
        "High score on wrong short prediction",
    ],
})
display(summary)                                               # show formatted table

,Failure Mode,Filter,Count,Signal
0,Knowledge-Gap,"EM=0, F1<0.3, score>0.3",146,"Confident wrong prediction, no token overlap"
1,Surface-Form,"EM=0, F1>0.4",0,Large EM–F1 gap in aggregate metrics
2,Distractor-Pull,"EM=0, pred len≤2, score>0.5",141,High score on wrong short prediction


---
## Section 7 — Pivot to the Learner Assignment

### What you just saw (SI demo)

- **Dataset:** TriviaQA `rc.nocontext` — open-domain trivia, no passage context.
- **Failure modes seen:** knowledge-gap dominant, surface-form mismatches, distractor pull.
- **Key takeaway:** The extractive reader is _not the right tool_ for no-context trivia — and that mismatch is exactly what makes the failure modes clear and diagnostic.

---

### What you will build (learner assignment)

Your dataset is a **curated tech / entertainment news QA set** — extractive, span-shaped questions, ~1,000 examples. Key structural differences:

| Dimension | SI Demo (TriviaQA) | Your Lab (News QA) |
|---|---|---|
| Context | None (question = context) | Full news article passage |
| Question style | Open-domain trivia | Span-extractive |
| Primary failure mode | Knowledge gap | Span-boundary errors |
| Expected EM | Low (no real context) | Higher |
| Expected F1–EM gap | Small | May be larger |

---

### Your tasks

1. **Load** your dataset and run the same `pipeline("question-answering")`.  
2. **Implement** `normalize_answer`, `exact_match`, and `token_f1` from scratch — do not copy from this notebook.  
3. **Run** the harness over all ~1,000 examples and save to `news_qa_results.csv`.  
4. **Report** mean EM, mean F1, and the F1–EM gap. Interpret the gap.  
5. **Identify three failure modes** from your own results — follow the same structure:
   - Name the pattern.
   - Show one concrete example (question, gold, prediction, scores).
   - State the production implication.

> **Important:** The three failure modes from this demo **cannot** be directly reused.  
> Your failure modes must emerge from your own dataset's predictions and reflect  
> the extractive reading-comprehension failure profile, not the no-context trivia profile.